# Knowledge Sources & RAG

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) CrewAI roadmap.

You will use `StringKnowledgeSource` at the crew level and at the agent level so agents can retrieve context during tasks.

In [ ]:
!pip install -q crewai crewai-tools

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## StringKnowledgeSource: crew-level and agent-level

**Crew-level** `knowledge_sources` are shared by every agent on that crew. **Agent-level** `knowledge_sources` are only available to that agent. Below, the support agent answers using shared company info plus its own product facts.

In [ ]:
from crewai import Agent, Crew, Process, Task
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource

company_info = StringKnowledgeSource(
    content=(
        "Our company was founded in 2020. We specialize in AI solutions for healthcare. "
        "Support hours are 9am–5pm ET."
    ),
)

product_facts = StringKnowledgeSource(
    content=(
        "Product name: HealthLens. Current version: 2.1. "
        "It ingests FHIR exports and produces clinician-friendly summaries."
    ),
)

support_agent = Agent(
    role="Support Specialist",
    goal="Answer customer questions using company and product knowledge",
    backstory="You cite only information that appears in your knowledge sources.",
    knowledge_sources=[product_facts],
    verbose=True,
)

answer_task = Task(
    description=(
        "Using your knowledge, answer: When was the company founded, "
        "what is the product called, and what does it do?"
    ),
    expected_output="A short factual answer with no invented details.",
    agent=support_agent,
)

crew = Crew(
    agents=[support_agent],
    tasks=[answer_task],
    process=Process.sequential,
    knowledge_sources=[company_info],
    verbose=True,
)

result = crew.kickoff()
print(result)

## Key takeaways

- **Crew knowledge** is shared across agents; **agent knowledge** is private to that agent.
- `StringKnowledgeSource` is the simplest way to try RAG-style retrieval without files.
- For file types (PDF, CSV, JSON, Excel, text), use the matching `*KnowledgeSource` class and paths relative to your `knowledge` directory.
- Tune retrieval with `KnowledgeConfig` (`results_limit`, `score_threshold`) and set `embedder` when you need a non-default embedding provider.